# 02 — Preparación de datos en Colab

Este notebook prepara el dataset para entrenar:
1. Monta Google Drive
2. Clona el repo (o lo copia desde Drive)
3. Instala dependencias
4. Descarga las imágenes desde los CSV (pants y tops)
5. Genera splits estratificados train/val/test

**Asume que ya subiste a Drive:**
- `data/preprocessed/pants_1.csv`
- `data/preprocessed/tops_1.csv`

Estructura sugerida en Drive:
```
/content/drive/MyDrive/master_ia/fashion-extraction/
├── data/preprocessed/
│   ├── pants_1.csv
│   └── tops_1.csv
└── (lo demás se crea automáticamente)
```

## 1. Mount Drive + clonar repo

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/master_ia/fashion-extraction'
REPO_DIR = '/content/fashion-feature-extraction'

# Opcion A: clonar desde GitHub (recomendado si tenes el repo en GitHub)
# !git clone <tu-repo-url> {REPO_DIR}

# Opcion B: copiar desde Drive (si tenes el codigo en Drive)
import shutil, os
SRC_IN_DRIVE = f'{DRIVE_ROOT}/code'  # ajusta si tu codigo esta en otro lado
if os.path.exists(SRC_IN_DRIVE) and not os.path.exists(REPO_DIR):
    shutil.copytree(SRC_IN_DRIVE, REPO_DIR)
    print(f'Copiado {SRC_IN_DRIVE} -> {REPO_DIR}')

%cd {REPO_DIR}
!ls


## 2. Instalar dependencias

In [ ]:
!pip install -q -r requirements.txt


## 3. Setup paths con Drive

In [ ]:
import sys
sys.path.insert(0, '/content/fashion-feature-extraction')

from src.data.colab import setup_colab
config = setup_colab('config/pipeline_config.yaml')
print('CSVs:')
print('  pants:', config['data']['pants_csv'])
print('  tops: ', config['data']['tops_csv'])
print('Imagenes:')
print('  pants:', config['paths']['images_pants'])
print('  tops: ', config['paths']['images_tops'])
print('Modelos:', config['paths']['models'])


## 4. Descargar imágenes (sample estratificado)

Descargamos ~15k imágenes de cada CSV. Tarda **~30-60 min** dependiendo
de la red. Las descargas son idempotentes (cache) y se pueden interrumpir
y reanudar sin perder progreso.


In [ ]:
SAMPLE_PANTS = 15000
SAMPLE_TOPS = 15000

from src.data.downloader import download_csv_images

log_pants = download_csv_images(
    csv_path=config['data']['pants_csv'],
    cache_dir=config['paths']['images_pants'],
    sample=SAMPLE_PANTS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/pants_download_log.csv",
)
print(f"\nPants OK: {log_pants['success'].sum()}/{len(log_pants)}")


In [ ]:
log_tops = download_csv_images(
    csv_path=config['data']['tops_csv'],
    cache_dir=config['paths']['images_tops'],
    sample=SAMPLE_TOPS,
    workers=8,
    timeout=15,
    log_path=f"{config['paths']['splits']}/tops_download_log.csv",
)
print(f"\nTops OK: {log_tops['success'].sum()}/{len(log_tops)}")


## 5. Splits estratificados (filtrando por imágenes descargadas)

In [ ]:
from src.data.splits import generate_splits

# Pants
pants_urls = set(log_pants[log_pants['success']]['url'])
pants_paths = generate_splits(
    csv_path=config['data']['pants_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=pants_urls,
)
print('Splits pants:', pants_paths)


In [ ]:
# Tops
tops_urls = set(log_tops[log_tops['success']]['url'])
tops_paths = generate_splits(
    csv_path=config['data']['tops_csv'],
    output_dir=config['paths']['splits'],
    stratify_col=config['data']['splits']['stratify_col'],
    train_ratio=config['data']['splits']['train_ratio'],
    val_ratio=config['data']['splits']['val_ratio'],
    test_ratio=config['data']['splits']['test_ratio'],
    min_samples_per_class=config['data']['splits']['min_samples_per_class'],
    seed=config['seed'],
    filter_urls=tops_urls,
)
print('Splits tops:', tops_paths)


## 6. Sanity check final

In [ ]:
import pandas as pd
from pathlib import Path

print('=== Resumen ===')
for ds in ['pants_1', 'tops_1']:
    for split in ['train', 'val', 'test']:
        p = Path(config['paths']['splits']) / f'{ds}_{split}.csv'
        if p.exists():
            df = pd.read_csv(p)
            print(f'  {ds}/{split}: {len(df):,} filas')

print('\n✓ Listo para entrenar. Ejecuta 03_train_pants_colab.ipynb')
